<table style="width: 100%;">
<tr>
<td style="width: 50%; text-align: right; vertical-align: middle;">
<img src="https://github.com/gitpizzanow/dummy-files/blob/main/tp3nlp.jpg?raw=true" width="150">
</td>
<td style="width: 50%; text-align: left; vertical-align: middle;">

##  (NLP)   | TF-IDF + Cosine Similarity
> [SERIE](https://tp3-nlp-ing4.netlify.app/)


* *Document Frequency (DF)*
* *IDF + Smoothing*
* *TF (Term Frequency)*
* *Cosine Similarity*



</td>
</tr>
</table>

>Data: Wikipedia-like dataset

In [1]:
from sklearn.datasets import fetch_20newsgroups

docs = fetch_20newsgroups(
    subset='train',
    remove=('headers', 'footers', 'quotes')
).data[:3000]

In [2]:
type(docs)

list

In [3]:
docs[19]

'QUESTION:\n  What is the EXACT entry (parameter and syntax please), in the X-Terminal\nconfiguration file (loaded when the X-Terminal boots), to add another system \nto the TCP/IP access control list?   \n\n  BACKGROUND:\n  I have two unix systems, 1. an AT&T 3B2 running X11R3 and MIT\'s X11R4 and \n2. a Sun SS10 without any X.  \n  I want to have a window to the Sun and the 3B2 on the NCD X-Terminal at the\nsame time.  I can do this if I manually set the Network Parameter TCP/IP\nAccess Control List to off, then login to my telnet session. Not Great!  \n  I\'ve tried to get "xhost" to work and failed.  Either my syntax is wrong\nor the X11R3 implementation is bogus.  \n  I am trying to edit the NCD configuration file that is loaded when the \nNCD boots.  No matter what entry I add or edit, the NCD still boots with\nthe TCP/IP Access Control list containing only the 3B2.\n  My manuals are worthless so any help would be most appreciated!!  Thanks!'

In [4]:
!pip install num2word

![STEP 1 - Preprocessing](https://img.shields.io/badge/STEP%201%20-%20Preprocessing-blue)

In [5]:
#comlete the code
import string
import num2word
def preprocess(text):
    text = text.lower()
    for p in string.punctuation:
        text = text.replace(p, '')
    result = []
    for token in text.split():
        if p.isdigit():
            result.append(num2word.word(p))
        else:
            result.append(token)
    return result

![STEP 2 - Vocabulary](https://img.shields.io/badge/STEP%202%20-%20Vocabulary-blue)

In [6]:
#comlete the code
def build_vocab(docs):
    print("Building vocab...")
    v = set()
    for doc in docs:
        v.update(preprocess(doc))
    v = sorted(v)
    print(f"  Vocab size: {len(v)}")
    return v

[![STEP 3 DF](https://img.shields.io/badge/STEP_3_DF-Document_Frequency-pink)](https://digitalpro.dev)

In [24]:
def compute_df(preprocessed_docs, vocab):
    vocab_set = set(vocab)
    df = {word: 0 for word in vocab}
    for doc_tokens in preprocessed_docs:
        unique_tokens = set(doc_tokens) & vocab_set
        for word in unique_tokens:
            df[word] += 1
    return df

[![STEP 4 IDF](https://img.shields.io/badge/STEP_4_IDF-Inverse_Document_Frequency-pink)](https://digitalpro.dev)

In [13]:
import math
def compute_idf(df, N):
    return {word: math.log((N+1)/(df[word]+1)) + 1 for word in df.keys()}

[![STEP 5 TF Vector](https://img.shields.io/badge/STEP_5_TF-Term_Frequency_Vector-pink)](https://digitalpro.dev)

In [8]:
def compute_tf(doc_tokens, vocab):
    n = len(doc_tokens)
    if n == 0:
        return {word: 0 for word in vocab}
    token_counts = {}
    for t in doc_tokens:
        token_counts[t] = token_counts.get(t, 0) + 1
    return {word: token_counts.get(word, 0) / n for word in vocab}

[![STEP 6 TF-IDF Matrix](https://img.shields.io/badge/STEP_6_TFIDF-Build_TF_IDF_Matrix-pink)](https://digitalpro.dev)

In [20]:
def build_tfidf(docs):
    tfidf_matrix = []
    vocab = build_vocab(docs)

    print("Preprocessing all docs...")
    preprocessed = [preprocess(doc) for doc in docs]

    print("Computing DF...")
    df = compute_df(preprocessed, vocab)
    print("Computing IDF...")
    idf = compute_idf(df, len(docs))

    print("Building TF-IDF matrix...")
    for i, doc_tokens in enumerate(preprocessed):
        if i % 500 == 0:
            print(f"  Doc {i}/{len(docs)}")
        tf = compute_tf(doc_tokens, vocab)
        vec = np.array([tf[w] * idf[w] for w in vocab])
        tfidf_matrix.append(vec)

    tfidf_matrix = np.array(tfidf_matrix)
    print(f"Matrix shape: {tfidf_matrix.shape}")
    return tfidf_matrix, vocab


[![STEP 7 Cosine Similarity](https://img.shields.io/badge/STEP_7-Cosine_Similarity-pink)](https://digitalpro.dev)

In [10]:
import numpy as np
def cosine(a, b):
  dot_product = np.dot(a, b)
  norm_a = np.linalg.norm(a)
  norm_b = np.linalg.norm(b)
  if norm_a == 0 or norm_b == 0:
        return 0.0
  return dot_product / (norm_a * norm_b)


[![STEP 8 Search Engine](https://img.shields.io/badge/STEP_8-Search_Engine_REAL_VERSION-orange)](https://digitalpro.dev)

In [11]:
import numpy as np
def search(query, docs, top_k=5):
    tfidf_matrix, vocab = build_tfidf(docs)

    q_vec = np.zeros(len(vocab)) # build query vector
    q_words = preprocess(query)

    for i, w in enumerate(vocab):
        q_vec[i] = q_words.count(w)

    if np.sum(q_vec) > 0:
        q_vec = q_vec / np.sum(q_vec)

    scores = []

    for i, doc_vec in enumerate(tfidf_matrix):
        score = cosine(q_vec, doc_vec)
        scores.append((score, i))

    scores.sort(reverse=True)

    return scores[:top_k]

> TEST

In [25]:
query = "machine learning neural network"
results = search(query, docs)

for score, idx in results:
    print(score)
    print(docs[idx][:200])
    print("-" * 50)

Building vocab...
  Vocab size: 44829
Preprocessing all docs...
Computing DF...
Computing IDF...
Building TF-IDF matrix...
  Doc 0/3000
  Doc 500/3000
  Doc 1000/3000
  Doc 1500/3000
  Doc 2000/3000
  Doc 2500/3000
Matrix shape: (3000, 44829)
0.1883923736864208
I just recently bought a 4 MB ram card for my original mac portable 
(backlit) and have since had some bizarre crashes. It happens when I put 
the machine to sleep and wake the machine up. sometimes i
--------------------------------------------------
0.16583730477514397
I'm looking for a Singer Featherweight 221 sewing machine (old, black 
sewing machine in black case).

Please contact:
--------------------------------------------------
0.14135585533489603

Yeah.  Sounds typical.  Windows makes all sorts of extra demands on hardware,
and therefore your machine can't keep up with things.  Ever notice how when
acessing the floppies in Windows, everything 
--------------------------------------------------
0.13138691132902652



T